In [30]:
# Importing Libraries

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

In [31]:
# Importing Dataset

df = pd.read_csv('loan_data.csv')

df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [32]:
# basic Exploration

print("Information : \n")
print(df.info())

print("\nMissing Values : \n")
print(df.isnull().sum())

print("\nDuplicate Entries : \n")
print(df.duplicated().sum())


Information : 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  object 
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  object 
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_de

In [33]:
# Splitting the Dataset
x = df.drop('loan_status',axis = 1)
y = df['loan_status']

x_train,x_test,y_train,y_test = train_test_split(x,y,
                                                random_state = 30,
                                                test_size = 0.2,
                                                stratify = y)

In [34]:
# Data Preprocessing

df['person_education'].unique()

categorical_cols = ['person_gender','person_home_ownership','loan_intent','previous_loan_defaults_on_file']

one_hot = ('onehot',OneHotEncoder(drop = 'if_binary',handle_unknown = 'ignore'),categorical_cols)
ordenco = ('ordinal',OrdinalEncoder(categories = [['Doctorate','Master','Bachelor','Associate','High School']]),['person_education'])

In [35]:
# Transformer

trf1 = ColumnTransformer(transformers = [
    one_hot,
    ordenco
],remainder = "passthrough")

In [36]:
# Importing Decison Tree Classifier

from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()

# Decison Tree Classifier

In [37]:
# Pipeline Creation 
pipe = make_pipeline(trf1,model)

In [38]:
# Training the Model 

pipe.fit(x_train,y_train)

,steps,"[('columntransformer', ...), ('decisiontreeclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehot', ...), ('ordinal', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [39]:
# Prediction and Evaluation

y_pred = pipe.predict(x_test)

In [41]:
from sklearn.metrics import accuracy_score,confusion_matrix

print("Accuracy Score : ",accuracy_score(y_pred,y_test))
print("Confusion Matrix : \n", confusion_matrix(y_pred,y_test))

Accuracy Score :  0.9004444444444445
Confusion Matrix : 
 [[6539  435]
 [ 461 1565]]


# Decision Tree Classifier - Hyperparameters

In [56]:
params = {
    'decisiontreeclassifier__max_depth' : [3,5,7,10],
    'decisiontreeclassifier__min_samples_split' : [2,10,20],
    'decisiontreeclassifier__min_samples_leaf' : [1,5,10]
}

In [57]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    pipe,
    param_grid = params,
    cv = 5,
    scoring = 'accuracy',
    n_jobs = -1
)

In [58]:
grid.fit(x_train,y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'decisiontreeclassifier__max_depth': [3, 5, ...], 'decisiontreeclassifier__min_samples_leaf': [1, 5, ...], 'decisiontreeclassifier__min_samples_split': [2, 10, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('onehot', ...), ('ordinal', ...)]"


In [59]:
y_pred2 = grid.predict(x_test)


print("Accuracy Score : ",accuracy_score(y_pred2,y_test))
print("Confusion Matrix : \n", confusion_matrix(y_pred2,y_test))

Accuracy Score :  0.9155555555555556
Confusion Matrix : 
 [[6819  579]
 [ 181 1421]]


In [60]:
grid.best_params_

{'decisiontreeclassifier__max_depth': 10,
 'decisiontreeclassifier__min_samples_leaf': 10,
 'decisiontreeclassifier__min_samples_split': 2}